In [1]:
from pathlib import Path
import geopandas as gpd

gpx_path = Path(r"C:\RaceGuard\data\raw\peachtree_course_plotaroute_2300594.gpx")

print("Path exists:", gpx_path.exists())

layers = gpd.list_layers(gpx_path)
print(layers)

Path exists: True
           name    geometry_type
0     waypoints            Point
1        routes       LineString
2        tracks  MultiLineString
3  route_points            Point
4  track_points            Point


In [2]:
course_tracks = gpd.read_file(
    gpx_path,
    layer="tracks",
)

geometry_column = course_tracks.geometry.name

attribute_data = course_tracks.drop(
    columns=geometry_column
)

populated_attributes = attribute_data.dropna(
    axis="columns",
    how="all",
)

print(f"Number of track rows: {len(course_tracks)}")
print(f"Columns: {course_tracks.columns.tolist()}")
print(f"CRS: {course_tracks.crs}")

print("\nGeometry types:")
print(
    course_tracks.geom_type
    .value_counts(dropna=False)
    .to_string()
)

print(
    "\nContains missing geometry:",
    course_tracks.geometry.isna().any(),
)

print(
    "Contains empty geometry:",
    course_tracks.geometry.is_empty.any(),
)

print(
    "All geometries valid:",
    course_tracks.geometry.is_valid.all(),
)

print("\nPopulated track attributes:")

if populated_attributes.shape[1] == 0:
    print("No populated non-geometry attributes")
else:
    print(
        populated_attributes.to_string(
            index=False
        )
    )

Number of track rows: 1
Columns: ['name', 'cmt', 'desc', 'src', 'link1_href', 'link1_text', 'link1_type', 'link2_href', 'link2_text', 'link2_type', 'number', 'type', 'geometry']
CRS: EPSG:4326

Geometry types:
MultiLineString    1

Contains missing geometry: False
Contains empty geometry: False
All geometries valid: True

Populated track attributes:
                   name
AJC Peachtree Road Race


In [3]:
OFFICIAL_DISTANCE_KM = 10.0
LENGTH_TOLERANCE_PERCENT = 1.0

metric_crs = course_tracks.estimate_utm_crs()

course_tracks_metric = course_tracks.to_crs(
    metric_crs
)

route_length_metres = (
    course_tracks_metric.geometry.length.sum()
)

route_length_km = route_length_metres / 1000
route_length_miles = route_length_km * 0.621371

difference_metres = abs(
    route_length_metres
    - OFFICIAL_DISTANCE_KM * 1000
)

difference_percent = (
    difference_metres
    / (OFFICIAL_DISTANCE_KM * 1000)
    * 100
)

length_check_passed = (
    difference_percent
    <= LENGTH_TOLERANCE_PERCENT
)

track_geometry = course_tracks.geometry.iloc[0]

if track_geometry.geom_type == "MultiLineString":
    track_parts = list(track_geometry.geoms)
else:
    track_parts = [track_geometry]

coordinate_count = sum(
    len(part.coords)
    for part in track_parts
)

print(f"Original CRS: {course_tracks.crs}")
print(f"Metric CRS: {metric_crs}")
print(f"Track parts: {len(track_parts)}")
print(f"Coordinate count: {coordinate_count:,}")
print(f"Calculated length: {route_length_metres:,.2f} m")
print(f"Calculated length: {route_length_km:.4f} km")
print(f"Calculated length: {route_length_miles:.4f} miles")
print(f"Difference from 10K: {difference_metres:.2f} m")
print(f"Percentage difference: {difference_percent:.3f}%")
print(f"Within {LENGTH_TOLERANCE_PERCENT:.1f}% tolerance: {length_check_passed}")

Original CRS: EPSG:4326
Metric CRS: EPSG:32616
Track parts: 1
Coordinate count: 155
Calculated length: 9,990.25 m
Calculated length: 9.9902 km
Calculated length: 6.2077 miles
Difference from 10K: 9.75 m
Percentage difference: 0.098%
Within 1.0% tolerance: True


In [4]:
course_line = track_parts[0]

start_coordinate = course_line.coords[0]
finish_coordinate = course_line.coords[-1]

start_longitude = start_coordinate[0]
start_latitude = start_coordinate[1]

finish_longitude = finish_coordinate[0]
finish_latitude = finish_coordinate[1]

minimum_longitude, minimum_latitude, maximum_longitude, maximum_latitude = (
    course_tracks.total_bounds
)

bounds_are_in_atlanta = (
    -85.0 <= minimum_longitude <= -84.0
    and -85.0 <= maximum_longitude <= -84.0
    and 33.0 <= minimum_latitude <= 34.5
    and 33.0 <= maximum_latitude <= 34.5
)

route_runs_generally_south = (
    start_latitude > finish_latitude
)

print("Start point:")
print(f"  Latitude: {start_latitude:.6f}")
print(f"  Longitude: {start_longitude:.6f}")

print("\nFinish point:")
print(f"  Latitude: {finish_latitude:.6f}")
print(f"  Longitude: {finish_longitude:.6f}")

print("\nCourse bounds:")
print(f"  Minimum longitude: {minimum_longitude:.6f}")
print(f"  Minimum latitude: {minimum_latitude:.6f}")
print(f"  Maximum longitude: {maximum_longitude:.6f}")
print(f"  Maximum latitude: {maximum_latitude:.6f}")

print(f"\nBounds fall within Atlanta region: {bounds_are_in_atlanta}")
print(f"Route runs generally north-to-south: {route_runs_generally_south}")

Start point:
  Latitude: 33.849322
  Longitude: -84.363280

Finish point:
  Latitude: 33.781792
  Longitude: -84.374667

Course bounds:
  Minimum longitude: -84.393950
  Minimum latitude: 33.781708
  Maximum longitude: -84.363232
  Maximum latitude: 33.849440

Bounds fall within Atlanta region: True
Route runs generally north-to-south: True


In [5]:
# import folium


# map_centre = [
#     (minimum_latitude + maximum_latitude) / 2,
#     (minimum_longitude + maximum_longitude) / 2,
# ]

# validation_map = folium.Map(
#     location=map_centre,
#     zoom_start=13,
#     tiles="OpenStreetMap",
# )

# folium.GeoJson(
#     course_tracks.__geo_interface__,
#     name="Candidate Peachtree course",
#     style_function=lambda feature: {
#         "color": "#1565C0",
#         "weight": 6,
#         "opacity": 0.9,
#     },
#     tooltip="Candidate Peachtree course",
# ).add_to(validation_map)

# folium.Marker(
#     location=[
#         start_latitude,
#         start_longitude,
#     ],
#     tooltip="Course start",
#     popup=(
#         f"Start<br>"
#         f"Latitude: {start_latitude:.6f}<br>"
#         f"Longitude: {start_longitude:.6f}"
#     ),
#     icon=folium.Icon(
#         color="green",
#         icon="play",
#     ),
# ).add_to(validation_map)

# folium.Marker(
#     location=[
#         finish_latitude,
#         finish_longitude,
#     ],
#     tooltip="Course finish",
#     popup=(
#         f"Finish<br>"
#         f"Latitude: {finish_latitude:.6f}<br>"
#         f"Longitude: {finish_longitude:.6f}"
#     ),
#     icon=folium.Icon(
#         color="red",
#         icon="flag",
#     ),
# ).add_to(validation_map)

# validation_map.fit_bounds(
#     [
#         [minimum_latitude, minimum_longitude],
#         [maximum_latitude, maximum_longitude],
#     ]
# )

# folium.LayerControl().add_to(
#     validation_map
# )

# validation_map

In [6]:
from pathlib import Path

import geopandas as gpd


# Locate the repository whether Jupyter started from the repository root
# or from the notebooks directory.
current_directory = Path.cwd()

repo_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)

output_path = (
    repo_root
    / "data"
    / "processed"
    / "peachtree_course.geojson"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)


# The validated GPX contains one MultiLineString with one component.
# Extract that component without silently discarding additional parts.
track_geometry = course_tracks.geometry.iloc[0]

if track_geometry.geom_type == "MultiLineString":
    track_parts = list(track_geometry.geoms)

    if len(track_parts) != 1:
        raise ValueError(
            f"Expected one connected track part, found {len(track_parts)}."
        )

    course_line = track_parts[0]
else:
    course_line = track_geometry


if course_line.geom_type != "LineString":
    raise ValueError(
        f"Expected a LineString, found {course_line.geom_type}."
    )


# Build RaceGuard's canonical course dataset.
# We deliberately retain EPSG:4326 because GeoJSON and FortyGuard use
# longitude/latitude coordinates.
canonical_course = gpd.GeoDataFrame(
    {
        "name": ["AJC Peachtree Road Race"],
        "source": ["PlotARoute route 2300594"],
        "validation_status": ["validated"],
    },
    geometry=[course_line],
    crs=course_tracks.crs,
)

canonical_course.to_file(
    output_path,
    driver="GeoJSON",
)


# Read the actual saved file back and validate it.
saved_course = gpd.read_file(output_path)

validation_checks = {
    "file exists": output_path.exists(),
    "one feature": len(saved_course) == 1,
    "LineString geometry": (
        saved_course.geometry.iloc[0].geom_type == "LineString"
    ),
    "EPSG:4326 CRS": saved_course.crs.to_epsg() == 4326,
    "valid geometry": saved_course.geometry.is_valid.all(),
    "non-empty geometry": not saved_course.geometry.is_empty.any(),
}

print("Saved course:", output_path)
print()

for check_name, passed in validation_checks.items():
    print(f"{check_name}: {passed}")

print()
print("Saved properties:")
print(saved_course.drop(columns="geometry").to_string(index=False))

if not all(validation_checks.values()):
    raise ValueError("The canonical course failed validation.")

Saved course: c:\RaceGuard\data\processed\peachtree_course.geojson

file exists: True
one feature: True
LineString geometry: True
EPSG:4326 CRS: True
valid geometry: True
non-empty geometry: True

Saved properties:
                   name                   source validation_status
AJC Peachtree Road Race PlotARoute route 2300594         validated


In [7]:
from pathlib import Path

import geopandas as gpd
import pandas as pd


current_directory = Path.cwd()

repo_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)

course_path = (
    repo_root
    / "data"
    / "processed"
    / "peachtree_course.geojson"
)

stations_path = (
    repo_root
    / "data"
    / "processed"
    / "peachtree_stations.csv"
)


# Load the validated course and project it into metre-based coordinates.
course = gpd.read_file(course_path)
metric_crs = course.estimate_utm_crs()
course_metric = course.to_crs(metric_crs)
course_line_metric = course_metric.geometry.iloc[0]
course_length_m = course_line_metric.length


# The official volunteer records identify paired aid stations
# at Miles 1 through 5.
MILES_TO_KM = 1.609344
official_miles = list(range(1, 6))

source_url = (
    "https://www.atlantatrackclub.org/"
    "northsidehospital-peachtree-road-race-"
    "volunteer-information-confidential"
)

stations = pd.DataFrame(
    {
        "station_id": [
            f"AID_{mile}"
            for mile in official_miles
        ],
        "official_mile": official_miles,
        "baseline_distance_km": [
            mile * MILES_TO_KM
            for mile in official_miles
        ],
        "side_count": 2,
        "has_water": True,
        "has_restrooms": True,
        "has_first_aid": True,
        "position_uncertainty_m": 200.0,
        "source_url": source_url,
    }
)


# Convert each route-relative distance into a geographic point.
station_geometries = [
    course_line_metric.interpolate(distance_km * 1000)
    for distance_km in stations["baseline_distance_km"]
]

station_points_metric = gpd.GeoDataFrame(
    stations.copy(),
    geometry=station_geometries,
    crs=metric_crs,
)

station_points_geo = station_points_metric.to_crs("EPSG:4326")

stations["longitude"] = station_points_geo.geometry.x
stations["latitude"] = station_points_geo.geometry.y

stations.to_csv(
    stations_path,
    index=False,
)


# Reload and validate what was actually saved.
saved_stations = pd.read_csv(stations_path)

saved_points_geo = gpd.GeoDataFrame(
    saved_stations.copy(),
    geometry=gpd.points_from_xy(
        saved_stations["longitude"],
        saved_stations["latitude"],
    ),
    crs="EPSG:4326",
)

saved_points_metric = saved_points_geo.to_crs(metric_crs)

distance_from_course_m = (
    saved_points_metric.geometry.distance(course_line_metric)
)

spacing_m = (
    saved_stations["baseline_distance_km"]
    .diff()
    .dropna()
    * 1000
)

service_columns = [
    "has_water",
    "has_restrooms",
    "has_first_aid",
]

station_checks = {
    "five logical stations": len(saved_stations) == 5,
    "unique station IDs": saved_stations["station_id"].is_unique,
    "ordered along course": (
        saved_stations["baseline_distance_km"]
        .is_monotonic_increasing
    ),
    "all stations inside course length": (
        (
            saved_stations["baseline_distance_km"] * 1000
            < course_length_m
        ).all()
    ),
    "mile conversion correct": (
        (
            saved_stations["baseline_distance_km"]
            - saved_stations["official_mile"] * MILES_TO_KM
        ).abs() < 1e-9
    ).all(),
    "two roadside setups each": (
        saved_stations["side_count"] == 2
    ).all(),
    "required services present": (
        saved_stations[service_columns].all().all()
    ),
    "derived points lie on course": (
        distance_from_course_m <= 0.01
    ).all(),
}

print("Saved stations:", stations_path)
print(f"Course length: {course_length_m:,.2f} m")
print(f"Minimum station spacing: {spacing_m.min():,.2f} m")
print()

for check_name, passed in station_checks.items():
    print(f"{check_name}: {passed}")

print()
print(
    saved_stations[
        [
            "station_id",
            "official_mile",
            "baseline_distance_km",
            "latitude",
            "longitude",
        ]
    ].to_string(index=False)
)

if not all(station_checks.values()):
    raise ValueError("Station baseline failed validation.")

Saved stations: c:\RaceGuard\data\processed\peachtree_stations.csv
Course length: 9,990.25 m
Minimum station spacing: 1,609.34 m

five logical stations: True
unique station IDs: True
ordered along course: True
all stations inside course length: True
mile conversion correct: True
two roadside setups each: True
required services present: True
derived points lie on course: True

station_id  official_mile  baseline_distance_km  latitude  longitude
     AID_1              1              1.609344 33.840821 -84.376145
     AID_2              2              3.218688 33.830282 -84.386687
     AID_3              3              4.828032 33.816425 -84.390109
     AID_4              4              6.437376 33.802904 -84.392963
     AID_5              5              8.046720 33.790818 -84.385256


In [8]:
AOI_BUFFER_M = 200.0

aoi_path = (
    repo_root
    / "data"
    / "processed"
    / "peachtree_course_aoi.geojson"
)


# Buffering 200 metres creates a corridor extending 200 metres
# from either side of the course.
aoi_geometry_metric = course_line_metric.buffer(
    AOI_BUFFER_M
)

aoi_metric = gpd.GeoDataFrame(
    {
        "name": ["Peachtree Road Race course corridor"],
        "buffer_metres": [AOI_BUFFER_M],
    },
    geometry=[aoi_geometry_metric],
    crs=metric_crs,
)

# FortyGuard accepts geographic GeoJSON coordinates.
aoi_geo = aoi_metric.to_crs("EPSG:4326")

aoi_geo.to_file(
    aoi_path,
    driver="GeoJSON",
)


# Reload the saved AOI and return it to metres for validation.
saved_aoi = gpd.read_file(aoi_path)
saved_aoi_metric = saved_aoi.to_crs(metric_crs)
saved_aoi_geometry = saved_aoi_metric.geometry.iloc[0]

aoi_area_km2 = saved_aoi_geometry.area / 1_000_000

aoi_checks = {
    "file exists": aoi_path.exists(),
    "one AOI feature": len(saved_aoi) == 1,
    "polygon geometry": (
        saved_aoi_geometry.geom_type
        in {"Polygon", "MultiPolygon"}
    ),
    "EPSG:4326 saved CRS": saved_aoi.crs.to_epsg() == 4326,
    "valid geometry": saved_aoi.geometry.is_valid.all(),
    "non-empty geometry": not saved_aoi.geometry.is_empty.any(),
    "entire course covered": (
        saved_aoi_geometry.covers(course_line_metric)
    ),
    "all stations covered": all(
        saved_aoi_geometry.covers(point)
        for point in saved_points_metric.geometry
    ),
}

print("Saved AOI:", aoi_path)
print(f"Buffer distance: {AOI_BUFFER_M:.0f} m per side")
print(f"AOI area: {aoi_area_km2:.3f} km²")
print()

for check_name, passed in aoi_checks.items():
    print(f"{check_name}: {passed}")

if not all(aoi_checks.values()):
    raise ValueError("Course AOI failed validation.")

Saved AOI: c:\RaceGuard\data\processed\peachtree_course_aoi.geojson
Buffer distance: 200 m per side
AOI area: 4.097 km²

file exists: True
one AOI feature: True
polygon geometry: True
EPSG:4326 saved CRS: True
valid geometry: True
non-empty geometry: True
entire course covered: True
all stations covered: True


In [9]:
from pathlib import Path
import json
import statistics
import sys

from dotenv import load_dotenv


# Find the repository root.
current_directory = Path.cwd()

repo_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)

sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / ".env")

from fortyguard import FortyGuardClient


aoi_path = (
    repo_root
    / "data"
    / "processed"
    / "peachtree_course_aoi.geojson"
)

cache_path = (
    repo_root
    / "data"
    / "raw"
    / "fortyguard"
    / "peachtree_heatmap_2026_07_04_0900_g100.json"
)


# Refuse to spend credits again if the response is already cached.
if cache_path.exists():
    raise FileExistsError(
        f"Cached response already exists at {cache_path}. "
        "Paid request cancelled."
    )


# Load and check the AOI before contacting FortyGuard.
with aoi_path.open("r", encoding="utf-8") as file:
    course_aoi = json.load(file)

if course_aoi.get("type") != "FeatureCollection":
    raise ValueError("AOI must be a GeoJSON FeatureCollection.")

if len(course_aoi.get("features", [])) != 1:
    raise ValueError("Expected exactly one AOI feature.")


client = FortyGuardClient()

print("Request ready:")
print("  Date       : 2026-07-04")
print("  Time       : 09:00")
print("  Filter type: 1 — single-hour snapshot")
print("  Granularity: 100 metres")
print("  Expected cost: approximately 4,220 credits")
print()


# This is the only paid operation in the cell.
response = client.create_heatmap(
    polygon_aoi=course_aoi,
    start_date="2026-07-04",
    start_time="09:00",
    filter_type=1,
    granularity=100,
)


# Cache immediately, before performing response analysis.
cache_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with cache_path.open("w", encoding="utf-8") as file:
    json.dump(response, file, indent=2)


# Perform a fast structural sanity check.
activity_id = response.get("activity_id")
result = response.get("result", {})
map_data = result.get("map_data", {})
features = map_data.get("features", [])

temperatures = [
    feature.get("properties", {}).get("average_temperature")
    for feature in features
]

temperatures = [
    temperature
    for temperature in temperatures
    if isinstance(temperature, (int, float))
]

geometry_types = {
    feature.get("geometry", {}).get("type")
    for feature in features
}

response_checks = {
    "activity ID returned": bool(activity_id),
    "result returned": isinstance(result, dict) and bool(result),
    "map is FeatureCollection": (
        map_data.get("type") == "FeatureCollection"
    ),
    "heatmap contains tiles": len(features) > 0,
    "tile geometries are polygons": (
        geometry_types <= {"Polygon", "MultiPolygon"}
    ),
    "every tile has temperature": (
        len(temperatures) == len(features)
        and len(features) > 0
    ),
}

print("Activity ID:", activity_id)
print("Cached response:", cache_path)
print(f"Cache size: {cache_path.stat().st_size:,} bytes")
print(f"Heatmap tiles: {len(features):,}")
print()

for check_name, passed in response_checks.items():
    print(f"{check_name}: {passed}")

if temperatures:
    print()
    print(f"Minimum temperature: {min(temperatures):.4f} °C")
    print(
        "Mean temperature: "
        f"{statistics.mean(temperatures):.4f} °C"
    )
    print(f"Maximum temperature: {max(temperatures):.4f} °C")

if not all(response_checks.values()):
    raise ValueError(
        "The response was cached, but failed structural validation."
    )

Request ready:
  Date       : 2026-07-04
  Time       : 09:00
  Filter type: 1 — single-hour snapshot
  Granularity: 100 metres
  Expected cost: approximately 4,220 credits

Submitted -> activity_id=741c32ee-7f05-4ca2-a31f-774a8770ec3e
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Activity ID: 741c32ee-7f05-4ca2-a31f-774a8770ec3e
Cached response: c:\RaceGuard\data\raw\fortyguard\peachtree_heatmap_2026_07_04_0900_g100.json
Cache size: 386,218 bytes
Heatmap tiles: 381

activity ID returned: True
result returned: True
map is FeatureCollection: True
heatmap contains tiles: True
tile geometries are polygons: True
every tile has temperature: True

Minimum temperature: 26.6293 °C
Mean temperature: 27.0199 °C
Maximum temperature: 27.3631 °C
